1. CARGAR DATOS EN UNA BBDD LLAMADA ECOMMERCE Y EN UNA COLECCION LLAMADA SALES 

In [4]:
from pymongo import MongoClient
import pandas as pd



In [7]:
#Utiliza la IP de tu anfitrión
cliente = MongoClient('mongos', 27017)

# comprobamos que es mongos
if cliente.admin.command("hello").get("msg") == "isdbgrid":
    print("OK: conectado a mongos")


#Creamos la instancia para interactuar con la colección dde sales
bbdd = cliente.ecommerce
coleccion_sales = bbdd.sales

OK: conectado a mongos


In [8]:

try:
    # Asegúrate de que el archivo productos.csv esté en la misma carpeta que el notebook
    df = pd.read_csv('ecommerce_sales_data.csv')
    
    # Convertimos el DataFrame a una lista de diccionarios para MongoDB
    datos = df.to_dict(orient='records')
    print(f"Archivo leído: {len(datos)} registros listos para importar.")
    
    # 3. Importación masiva
    resultado = coleccion_sales.insert_many(datos)
    print(f"Éxito: Se han insertado {len(resultado.inserted_ids)} documentos.")
    
except FileNotFoundError:
    print("Error: No se encontró el archivo 'productos.csv'.")
except Exception as e:
    print(f"Error durante la importación: {e}")

# 4. Comprobación de los primeros 10 documentos
print("\n--- MUESTRA DE LOS PRIMEROS 10 DOCUMENTOS ---")
cursor = coleccion_sales.find().limit(10)
for i, doc in enumerate(cursor, 1):
    print(f"{i}: {doc}")

Archivo leído: 3500 registros listos para importar.
Éxito: Se han insertado 3500 documentos.

--- MUESTRA DE LOS PRIMEROS 10 DOCUMENTOS ---
1: {'_id': ObjectId('69808e61b301c2e3bb901865'), 'Order Date': '2024-12-31', 'Product Name': 'Printer', 'Category': 'Office', 'Region': 'North', 'Quantity': 4, 'Sales': 3640, 'Profit': 348.93}
2: {'_id': ObjectId('69808e61b301c2e3bb901866'), 'Order Date': '2022-11-27', 'Product Name': 'Mouse', 'Category': 'Accessories', 'Region': 'East', 'Quantity': 7, 'Sales': 1197, 'Profit': 106.53}
3: {'_id': ObjectId('69808e61b301c2e3bb901867'), 'Order Date': '2022-05-11', 'Product Name': 'Tablet', 'Category': 'Electronics', 'Region': 'South', 'Quantity': 5, 'Sales': 5865, 'Profit': 502.73}
4: {'_id': ObjectId('69808e61b301c2e3bb901868'), 'Order Date': '2024-03-16', 'Product Name': 'Mouse', 'Category': 'Accessories', 'Region': 'South', 'Quantity': 2, 'Sales': 786, 'Profit': 202.87}
5: {'_id': ObjectId('69808e61b301c2e3bb901869'), 'Order Date': '2022-09-10', 'Pr

In [21]:
df.head()

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53
2,2022-05-11,Tablet,Electronics,South,5,5865,502.73
3,2024-03-16,Mouse,Accessories,South,2,786,202.87
4,2022-09-10,Mouse,Accessories,West,1,509,103.28


![img1](./img/img1.png)

2. Sharderar la bbdd y la coleccion cp

![img2](./img/img2.png)
![img2](./img/img3.png)

3. Contar nº documentos jp

![img1](./img/img4.png)

4. Listar las bbdd del cluster

In [14]:
bbdds= cliente.list_database_names()
print(bbdds)

['admin', 'config', 'ecommerce']


5. Muestra todos los productos category = 'office' y cuentalos 

In [20]:
print("\n--- OFFICE CATEGORY ---")
categ_office = coleccion_sales.find({"Category": "Office"})
for p in categ_office:
    print(f"Producto: {p['Product Name']}")


--- OFFICE CATEGORY ---
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Producto: Printer
Pro

6. Crear un usuario llamado 'Paco' con la password:  '1234' permisos de lectura y escrituta ademas comprobar que se conecte 

In [25]:

admin_db = cliente.admin
try:
    admin_db.command(
        "createUser", "Paco",
        pwd="1234",
        roles=[
            {"role": "readWriteAnyDatabase", "db": "admin"},
            {"role": "userAdminAnyDatabase", "db": "admin"}
        ]
    )
    print("Usuario 'Paco' creado correctamente.")

except Exception as e:
        print(f"Error al crear usuario: {e}")

Usuario 'Paco' creado correctamente.


Para comprobar que se conecte 

In [26]:
try:
    cliente_paco = MongoClient(
        'mongos', 
        27017, 
        username='Paco', 
        password='1234', 
        authSource='admin'
    )
    cliente_paco.admin.command('ping')
    print("Paco se ha conectado correctamente.")
    

except Exception as e:
    print(f"Error de conexión: {e}")

Paco se ha conectado correctamente.
